# SER Ablation Study — Colab Runner

Runs the four ablation variants on RAVDESS and generates comparison plots.

| Variant | Config | run_name |
|---|---|---|
| Baseline | crnn_ravdess.yaml | ravdess_baseline |
| +Harmonic | harmonic_only_ravdess.yaml | ravdess_harmonic |
| +FreqPos | freqpos_only_ravdess.yaml | ravdess_freqpos |
| +Both | harmonic_freqpos.yaml | ravdess_harmonic_freqpos |

**Your friend runs Baseline. You run the other three.**

To combine results: both of you save `runs/` and `results/` to Drive (cell 4),
then copy each other's results into `results/` before running the plot cell.

In [ ]:
# ── Cell 1: clone repo & install deps ─────────────────────────────────────
import os

REPO_URL = "https://github.com/YOUR_USERNAME/crnn-ser"  # ← update this
REPO_DIR = "/content/crnn-ser"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────────
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("No GPU — training will be slow but will still run.")

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Folder structure on Drive:
#   MyDrive/cs231n-ser/
#     data/ravdess/   ← put train.npz, val.npz, test.npz here
#     runs/           ← checkpoints saved here
#     results/        ← evaluation JSONs and plots saved here

DRIVE_ROOT = "/content/drive/MyDrive/cs231n-ser"
os.makedirs(f"{DRIVE_ROOT}/data/ravdess", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/results", exist_ok=True)
print("Drive mounted at", DRIVE_ROOT)

In [ ]:
# ── Cell 4: Configure paths ───────────────────────────────────────────────
# Data: preprocessed .npz files (train.npz, val.npz, test.npz)
# Option A — files already on Drive:
DATA_ROOT = f"{DRIVE_ROOT}/data/ravdess"

# Option B — copy from Drive to local SSD for faster I/O during training
# (uncomment if training is I/O bound)
# LOCAL_DATA = "/content/data/ravdess"
# os.makedirs(LOCAL_DATA, exist_ok=True)
# !cp -r {DRIVE_ROOT}/data/ravdess/* {LOCAL_DATA}/
# DATA_ROOT = LOCAL_DATA

# Verify data is present
for split in ["train", "val", "test"]:
    path = f"{DATA_ROOT}/{split}.npz"
    exists = os.path.exists(path)
    print(f"  {split}.npz: {'OK' if exists else 'MISSING — upload to Drive first'}")

In [ ]:
# ── Cell 5: Patch configs to point at local data_root ─────────────────────
import yaml

VARIANTS = [
    ("crnn_ravdess.yaml",            "ravdess_baseline"),
    ("harmonic_only_ravdess.yaml",   "ravdess_harmonic"),
    ("freqpos_only_ravdess.yaml",    "ravdess_freqpos"),
    ("harmonic_freqpos.yaml",        "ravdess_harmonic_freqpos"),
]

RUNTIME_CONFIGS = {}
os.makedirs("/content/runtime_configs", exist_ok=True)

for cfg_file, run_name in VARIANTS:
    with open(f"configs/{cfg_file}") as f:
        cfg = yaml.safe_load(f)
    cfg["data_root"] = DATA_ROOT
    cfg["run_name"] = run_name  # checkpoint dir: runs/{run_name}/
    out_path = f"/content/runtime_configs/{run_name}.yaml"
    with open(out_path, "w") as f:
        yaml.dump(cfg, f)
    RUNTIME_CONFIGS[run_name] = out_path
    print(f"  {run_name}: data_root={DATA_ROOT}")

In [ ]:
# ── Cell 6: CPU unit tests (run before GPU training) ──────────────────────
!python tests/test_components.py

In [ ]:
# ── Cell 7: GPU smoke train — 2 epochs, confirm loss drops ────────────────
# Uses the +Both config as the most complex path.
import yaml
smoke_cfg_path = "/content/runtime_configs/ravdess_harmonic_freqpos.yaml"
with open(smoke_cfg_path) as f:
    scfg = yaml.safe_load(f)
scfg["train"]["epochs"] = 2
smoke_out = "/content/runtime_configs/smoke.yaml"
with open(smoke_out, "w") as f:
    yaml.dump(scfg, f)

!python train.py --config {smoke_out}
print("\nSmoke train complete — if loss printed above, GPU path is working.")

## Full Ablation Training

Run the cells below for each variant you're responsible for.
**Baseline** is your friend's job — skip that cell and collect their checkpoint later.

Each run saves a checkpoint to `runs/{run_name}/best.pt`.
After training, the next section syncs everything to Drive.

In [ ]:
# ── Cell 8a: Train — Baseline (your friend runs this) ─────────────────────
# !python train.py --config {RUNTIME_CONFIGS['ravdess_baseline']}

In [ ]:
# ── Cell 8b: Train — +Harmonic only ───────────────────────────────────────
!python train.py --config {RUNTIME_CONFIGS['ravdess_harmonic']}

In [ ]:
# ── Cell 8c: Train — +FreqPos only ────────────────────────────────────────
!python train.py --config {RUNTIME_CONFIGS['ravdess_freqpos']}

In [ ]:
# ── Cell 8d: Train — +Both ────────────────────────────────────────────────
!python train.py --config {RUNTIME_CONFIGS['ravdess_harmonic_freqpos']}

In [ ]:
# ── Cell 9: Save checkpoints to Drive ─────────────────────────────────────
import shutil
for run_name in ["ravdess_harmonic", "ravdess_freqpos", "ravdess_harmonic_freqpos"]:
    src = f"runs/{run_name}"
    dst = f"{DRIVE_ROOT}/runs/{run_name}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"  Saved runs/{run_name} → Drive")
    else:
        print(f"  Skipped {run_name} (not trained yet)")

## Evaluation

Before running, copy your friend's baseline checkpoint from Drive:
```python
os.makedirs("runs/ravdess_baseline", exist_ok=True)
!cp "{DRIVE_ROOT}/runs/ravdess_baseline/best.pt" runs/ravdess_baseline/best.pt
```

In [ ]:
# ── Cell 10: Evaluate all trained variants ─────────────────────────────────
for run_name, cfg_path in RUNTIME_CONFIGS.items():
    ckpt = f"runs/{run_name}/best.pt"
    if not os.path.exists(ckpt):
        print(f"  SKIP {run_name}: no checkpoint at {ckpt}")
        continue
    print(f"\n{'='*60}\nEvaluating: {run_name}\n{'='*60}")
    !python evaluate.py --config {cfg_path} --checkpoint {ckpt}

In [ ]:
# ── Cell 11: Sync results to Drive ────────────────────────────────────────
for run_name in RUNTIME_CONFIGS:
    src = f"results/{run_name}"
    dst = f"{DRIVE_ROOT}/results/{run_name}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"  Saved results/{run_name} → Drive")

## Comparison Plots

Collect `results/` from both you and your friend, merge them under `results/`, then run the cell below.

In [ ]:
# ── Cell 12: Generate comparison plots ────────────────────────────────────
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

HOP_MS = 10.0  # hop_length_ms from config

VARIANTS_PLOT = [
    ("Baseline",   "ravdess_baseline",             "#2c3e50", "-"),
    ("+Harmonic",  "ravdess_harmonic",              "#e74c3c", "--"),
    ("+FreqPos",   "ravdess_freqpos",               "#3498db", "-."),
    ("+Both",      "ravdess_harmonic_freqpos",       "#27ae60", ":"),
]

loaded = {}
for label, run_name, color, ls in VARIANTS_PLOT:
    path = f"results/{run_name}/metrics.json"
    if os.path.exists(path):
        with open(path) as f:
            loaded[label] = (json.load(f), color, ls)
    else:
        print(f"  Missing: {path} — skipping {label}")

if not loaded:
    raise RuntimeError("No results found — run evaluation cells first.")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle("Ablation Study: Harmonic Dilation + Frequency Positional Encoding",
             fontsize=14, fontweight="bold", y=1.02)

# ── Left: latency-accuracy curves ────────────────────────────────────────
ax = axes[0]
for label, (metrics, color, ls) in loaded.items():
    curve = np.array(metrics["latency_curve"])
    t_ms  = np.arange(len(curve)) * HOP_MS
    uar   = metrics["uar"]
    ax.plot(t_ms, curve, color=color, linestyle=ls, linewidth=2.2,
            label=f"{label}  (UAR={uar:.3f})")

ax.set_xlabel("Elapsed time (ms)", fontsize=12)
ax.set_ylabel("Frame accuracy", fontsize=12)
ax.set_title("Latency–Accuracy Curve", fontsize=13)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.25)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

# Annotate 500 ms mark
ax.axvline(500, color="grey", linestyle=":", linewidth=1, alpha=0.6)
ax.text(510, 0.05, "500 ms", color="grey", fontsize=9)

# ── Right: UAR bar chart ──────────────────────────────────────────────────
ax2 = axes[1]
names  = list(loaded.keys())
uars   = [loaded[n][0]["uar"] for n in names]
colors = [loaded[n][1] for n in names]

bars = ax2.bar(names, uars, color=colors, alpha=0.85,
               edgecolor="white", linewidth=1.5, width=0.55)
for bar, u in zip(bars, uars):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.008,
             f"{u:.3f}", ha="center", va="bottom",
             fontsize=11, fontweight="bold")

ax2.set_ylabel("UAR (Unweighted Avg Recall)", fontsize=12)
ax2.set_title("Final UAR by Variant", fontsize=13)
ax2.set_ylim(0, 1.0)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax2.grid(True, alpha=0.25, axis="y")
ax2.tick_params(axis="x", labelsize=11)

plt.tight_layout()
os.makedirs("results", exist_ok=True)
out_path = "results/ablation_comparison.png"
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")

In [ ]:
# ── Cell 13: Confusion matrix grid ────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

EMOTIONS = ["happy", "sad", "angry", "neutral"]

n = len(loaded)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4.5))
if n == 1:
    axes = [axes]

for ax, (label, (metrics, color, _)) in zip(axes, loaded.items()):
    cm = np.array(metrics["confusion_matrix"], dtype=float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)  # row-normalize

    im = ax.imshow(cm_norm, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(EMOTIONS)))
    ax.set_yticks(range(len(EMOTIONS)))
    ax.set_xticklabels(EMOTIONS, rotation=30, ha="right", fontsize=9)
    ax.set_yticklabels(EMOTIONS, fontsize=9)
    ax.set_xlabel("Predicted", fontsize=10)
    ax.set_ylabel("True", fontsize=10)
    ax.set_title(f"{label}\nUAR={metrics['uar']:.3f}", fontsize=11)

    for i in range(len(EMOTIONS)):
        for j in range(len(EMOTIONS)):
            v = cm_norm[i, j]
            ax.text(j, i, f"{v:.2f}",
                    ha="center", va="center",
                    color="white" if v > 0.55 else "black",
                    fontsize=9)

fig.suptitle("Confusion Matrices (row-normalized recall)", fontsize=13, y=1.02)
plt.tight_layout()
out_path = "results/confusion_matrices.png"
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")

In [ ]:
# ── Cell 14: Save plots to Drive ──────────────────────────────────────────
for fname in ["ablation_comparison.png", "confusion_matrices.png"]:
    src = f"results/{fname}"
    dst = f"{DRIVE_ROOT}/results/{fname}"
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"  Saved {fname} → Drive")